In [ ]:
# KALMAN OPEN POLICY CONTRACT SOLVER v2.6 — ONE CELL / READ ONLY
# Goal: recover legacy trigger predicates and non-trigger baseline contract without changing canonical files.
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
from pathlib import Path
import pandas as pd, numpy as np, itertools, json, re

ROOT=Path("/content/drive/MyDrive/US_ETF/model_lab_v1/results")
AUD=ROOT/"open_revalidation_v1/open_revalidation_trade_audit.parquet"
LED=ROOT/"exit_policy_v1_0_pre2026/exit_policy_v1_0_1_trade_ledger.parquet"
assert AUD.exists() and LED.exists(), (AUD,LED)
a=pd.read_parquet(AUD); l=pd.read_parquet(LED)
print("[AUDIT]",a.shape,"[LEDGER]",l.shape)

policies=sorted({c[:-9] for c in a.columns if c.startswith("OPEN_") and c.endswith("__trigger")})
print("[POLICIES]",policies)

# ---- 1. Trigger predicate discovery: exact/simple predicates only ----
features=["position_return_prev_close","position_return_open","position_return_5m","position_return_15m",
          "overnight_gap_return","open_momentum_5m","open_momentum_15m","giveback_prev_close_to_5m"]
features=[c for c in features if c in a.columns]
thresholds=[0.0,-0.0005,-0.001,-0.002,-0.003,-0.005,-0.0075,-0.01,-0.015,-0.02,-0.03,
            0.0005,0.001,0.002,0.003,0.005,0.0075,0.01,0.015,0.02,0.03]

atoms=[]
for c in features:
    x=pd.to_numeric(a[c],errors="coerce")
    for t in thresholds:
        atoms.append((f"{c}<={t:g}",(x<=t).fillna(False)))
        atoms.append((f"{c}>={t:g}",(x>=t).fillna(False)))

def score(y,p):
    y=np.asarray(y,bool); p=np.asarray(p,bool)
    return dict(agree=int((y==p).sum()), fp=int((~y&p).sum()), fn=int((y&~p).sum()),
                actual=int(y.sum()), candidate=int(p.sum()))

print("\n[TRIGGER SOLVER]")
best_contract={}
for pol in policies:
    y=a[f"{pol}__trigger"].fillna(False).astype(bool)
    ranked=[]
    for name,p in atoms:
        s=score(y,p); ranked.append((s["fp"]+s["fn"],name,s,p))
    # AND/OR of top 80 single atoms: enough to discover threshold+confirmation style contracts.
    seed=sorted(ranked,key=lambda z:(z[0],abs(z[2]["candidate"]-z[2]["actual"])))[:80]
    for i,(e1,n1,s1,p1) in enumerate(seed):
        for e2,n2,s2,p2 in seed[i+1:]:
            for op in ("AND","OR"):
                p=(p1&p2) if op=="AND" else (p1|p2)
                s=score(y,p)
                ranked.append((s["fp"]+s["fn"],f"({n1}) {op} ({n2})",s,p))
    ranked=sorted(ranked,key=lambda z:(z[0],abs(z[2]["candidate"]-z[2]["actual"])))
    print("\n",pol,"actual_true=",int(y.sum()))
    for e,n,s,p in ranked[:10]: print(" err=",e,n,s)
    exact=[z for z in ranked if z[0]==0]
    best_contract[pol]=exact[0][1] if exact else None
    print(" EXACT=",best_contract[pol])

# ---- 2. Historical OPEN ledger inventory / direct join ----
op=l[l["policy"].astype(str).str.startswith("OPEN_")].copy()
fx=l[l["policy"].astype(str).eq("FIXED_4")].copy()
print("\n[LEDGER POLICY COUNTS]")
print(l["policy"].value_counts().to_string())

keys=[k for k in ["fold","symbol","entry_timestamp","entry_seq"] if k in l.columns and k in a.columns]
print("[JOIN KEYS]",keys)

# audit row identity is FIXED_4 lineage; join fixed ledger first, then OPEN policies by same entry identity.
af=a.copy()
for c in ["entry_timestamp","exit_timestamp"]:
    if c in af.columns: af[c]=pd.to_datetime(af[c],utc=True,errors="coerce")
for df in [op,fx]:
    for c in ["entry_timestamp","exit_timestamp"]:
        if c in df.columns: df[c]=pd.to_datetime(df[c],utc=True,errors="coerce")

fxcols=keys+[c for c in ["gross_return","net_return","exit_price","exit_timestamp","exit_reason","holding_bars","weight"] if c in fx.columns]
fxu=fx[fxcols].drop_duplicates(keys)
base=af.merge(fxu,on=keys,how="left",suffixes=("","__ledger_fixed"))
print("\n[FIXED LEDGER MATCH]",base[[c for c in base.columns if c.endswith("__ledger_fixed")]].notna().any(axis=1).sum(),"/",len(base))

print("\n[NON-TRIGGER BASELINE CONTRACT BY POLICY]")
for pol in policies:
    sub=op[op["policy"].astype(str).eq(pol)].copy()
    cols=keys+[c for c in ["gross_return","net_return","exit_price","exit_timestamp","exit_reason","holding_bars","weight"] if c in sub.columns]
    sub=sub[cols].drop_duplicates(keys)
    m=base.merge(sub,on=keys,how="left",suffixes=("","__open"))
    trig=m[f"{pol}__trigger"].fillna(False).astype(bool)
    nt=m[~trig].copy()
    print("\n",pol,"open_ledger_matches=",len(sub),"nontrigger_rows=",len(nt))
    # Compare stored audit OPEN net, OPEN ledger net, historical FIXED ledger net, reconstructed fixed4.
    audnet=f"{pol}__net_return"
    candidates={}
    if audnet in nt: candidates["audit_open_net"]=pd.to_numeric(nt[audnet],errors="coerce")
    if "net_return__open" in nt: candidates["ledger_open_net"]=pd.to_numeric(nt["net_return__open"],errors="coerce")
    if "net_return__ledger_fixed" in nt: candidates["ledger_fixed_net"]=pd.to_numeric(nt["net_return__ledger_fixed"],errors="coerce")
    if "reconstructed_fixed4_net_return" in nt: candidates["reconstructed_fixed4"]=pd.to_numeric(nt["reconstructed_fixed4_net_return"],errors="coerce")
    names=list(candidates)
    for i in range(len(names)):
        for j in range(i+1,len(names)):
            x,y=candidates[names[i]],candidates[names[j]]
            ok=x.notna()&y.notna()
            if ok.any():
                d=(x[ok]-y[ok]).abs()
                print(names[i],"vs",names[j],"n=",int(ok.sum()),"max=",float(d.max()),"median=",float(d.median()))

# ---- 3. Triggered rows: verify audit vs historical OPEN ledger directly ----
print("\n[TRIGGERED AUDIT↔LEDGER CONTRACT]")
for pol in policies:
    sub=op[op["policy"].astype(str).eq(pol)].copy()
    if sub.empty: print(pol,"NO LEDGER ROWS"); continue
    cols=keys+[c for c in ["net_return","gross_return","exit_price","exit_timestamp","exit_reason","holding_bars"] if c in sub.columns]
    m=af.merge(sub[cols].drop_duplicates(keys),on=keys,how="left",suffixes=("","__ledger_open"))
    tr=m[m[f"{pol}__trigger"].fillna(False).astype(bool)]
    print("\n",pol,"trigger_n=",len(tr))
    if len(tr)==0: continue
    for ac,lc in [(f"{pol}__net_return","net_return__ledger_open"),(f"{pol}__exit_price","exit_price__ledger_open")]:
        if ac in tr and lc in tr:
            x=pd.to_numeric(tr[ac],errors="coerce"); y=pd.to_numeric(tr[lc],errors="coerce"); ok=x.notna()&y.notna()
            print(ac,"vs",lc,"n=",int(ok.sum()),"max_abs_err=",float((x[ok]-y[ok]).abs().max()) if ok.any() else None)

print("\nREAD ONLY. No canonical file modified.")
print("NEXT: exact trigger predicates + exact non-trigger ledger contract are required before v2.7 rebuild.")
